# Phase 2 RV Feature Inspection

This notebook is inspection-only. It loads the frozen realized-variance outputs for US and India, checks the Phase 2 panel structure, and previews the generated diagnostics.

Production feature construction stays in `src/vrp/` and `scripts/build_features.py`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

DATA_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_TABLE_DIR = PROJECT_ROOT / 'reports' / 'tables'
REPORT_FIGURE_DIR = PROJECT_ROOT / 'reports' / 'figures'

US_RV_PATH = DATA_PROCESSED_DIR / 'us_rv.parquet'
INDIA_RV_PATH = DATA_PROCESSED_DIR / 'india_rv.parquet'
RV_SUMMARY_PATH = REPORT_TABLE_DIR / 'rv_summary.csv'
RV_CORRELATIONS_PATH = REPORT_TABLE_DIR / 'rv_estimator_correlations.csv'
RV_METADATA_PATH = REPORT_TABLE_DIR / 'rv_metadata.json'


In [ ]:
us_rv = pd.read_parquet(US_RV_PATH)
india_rv = pd.read_parquet(INDIA_RV_PATH)

us_rv.head()

In [ ]:
{
    'us_columns': us_rv.columns.tolist(),
    'india_columns': india_rv.columns.tolist(),
    'us_rows': len(us_rv),
    'india_rows': len(india_rv),
}

,date,open,high,low,close,adj_close,volume,source,market,symbol
0,1990-01-02,353.399994,359.690002,351.980011,359.690002,359.690002,162070000,yahoo_spx,US,^GSPC
1,1990-01-03,359.690002,360.589996,357.890015,358.760010,358.760010,192330000,yahoo_spx,US,^GSPC
2,1990-01-04,358.760010,358.760010,352.890015,355.670013,355.670013,177000000,yahoo_spx,US,^GSPC
3,1990-01-05,355.670013,355.670013,351.350006,352.200012,352.200012,158530000,yahoo_spx,US,^GSPC
4,1990-01-08,352.200012,354.239990,350.540009,353.790009,353.790009,140110000,yahoo_spx,US,^GSPC


In [ ]:
inspection = pd.DataFrame(
    {
        'market': ['US', 'INDIA'],
        'rv_gk_22d_ann_first_valid_index': [
            us_rv['rv_gk_22d_ann'].first_valid_index(),
            india_rv['rv_gk_22d_ann'].first_valid_index(),
        ],
        'rv_yz_22d_ann_first_valid_index': [
            us_rv['rv_yz_22d_ann'].first_valid_index(),
            india_rv['rv_yz_22d_ann'].first_valid_index(),
        ],
        'rv_cc_daily_first_value_is_nan': [
            pd.isna(us_rv.loc[0, 'rv_cc_daily']),
            pd.isna(india_rv.loc[0, 'rv_cc_daily']),
        ],
    }
)

inspection

['date',
 'market',
 'symbol',
 'log_return',
 'simple_return',
 'gap_return',
 'intraday_return',
 'rv_cc_daily',
 'rv_parkinson_daily',
 'rv_gk_daily',
 'rv_rs_daily',
 'rv_cc_22d_ann',
 'rv_parkinson_22d_ann',
 'rv_gk_22d_ann',
 'rv_rs_22d_ann',
 'rv_yz_22d_ann']

In [ ]:
us_rv[[
    'date',
    'rv_cc_daily',
    'rv_parkinson_daily',
    'rv_gk_daily',
    'rv_rs_daily',
    'rv_gk_22d_ann',
    'rv_yz_22d_ann',
]].head(30)

# Repeat the same preview for India to compare the panels side by side.
india_rv[[
    'date',
    'rv_cc_daily',
    'rv_parkinson_daily',
    'rv_gk_daily',
    'rv_rs_daily',
    'rv_gk_22d_ann',
    'rv_yz_22d_ann',
]].head(30)

,date,rv_cc_daily,rv_parkinson_daily,rv_gk_daily,rv_rs_daily,rv_gk_22d_ann,rv_yz_22d_ann
0,1990-01-02,NaN,0.000169,0.000115,0.000087,NaN,NaN
1,1990-01-03,6.702339e-06,0.000020,0.000026,0.000025,NaN,NaN
2,1990-01-04,7.482762e-05,0.000098,0.000107,0.000129,NaN,NaN
3,1990-01-05,9.612119e-05,0.000054,0.000038,0.000030,NaN,NaN
4,1990-01-08,2.028881e-05,0.000040,0.000047,0.000051,NaN,NaN
5,1990-01-09,1.405814e-04,0.000061,0.000029,0.000013,NaN,NaN
6,1990-01-10,4.394484e-05,0.000084,0.000100,0.000132,NaN,NaN
7,1990-01-11,1.229597e-05,0.000024,0.000028,0.000037,NaN,NaN
8,1990-01-12,6.242309e-04,0.000249,0.000104,0.000034,NaN,NaN
9,1990-01-15,7.493953e-05,0.000036,0.000021,0.000013,NaN,NaN


In [ ]:
missing_report = pd.DataFrame(
    {
        'market': ['US', 'INDIA'],
        'rv_gk_22d_ann_missing': [
            us_rv['rv_gk_22d_ann'].isna().sum(),
            india_rv['rv_gk_22d_ann'].isna().sum(),
        ],
        'rv_yz_22d_ann_missing': [
            us_rv['rv_yz_22d_ann'].isna().sum(),
            india_rv['rv_yz_22d_ann'].isna().sum(),
        ],
    }
)

summary = pd.read_csv(RV_SUMMARY_PATH)
correlations = pd.read_csv(RV_CORRELATIONS_PATH)
metadata = pd.read_json(RV_METADATA_PATH, typ='series')

{
    'missing_report': missing_report,
    'summary_markets': summary['market'].unique().tolist(),
    'correlation_rows': len(correlations),
    'metadata_phase': metadata.get('phase'),
}


Saved RV panel to: c:\Users\daksh\Desktop\EPAT\EPAT_Project_VRP_Regime\data\processed\us_rv.parquet
Rows: 9,160
Primary column first valid index: 21
